# Day 6: Introduction to GenAI + Prompt Engineering

1. Messy Invoice Text
2. PROMPT ENGINEERING
3. GROQ API
4. JSON PARSING
5. PANDAS DATAFRAME
6. ANALYSIS

**Tools:** Groq API, Python groq library, json, Pandas

In [1]:
! pip install groq --quiet              # Install groq python package

import os                               # working with files, folders and env
import json                             # Handling JSON data
import re                               # regular expressions for pattern matching
import time                             # time-related functions
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print('Libraries ready!')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 2.6 MB/s eta 0:00:00
Libraries ready!


## Talk to an LLM from Python using Groq

In [ ]:

from groq import Groq                                                 # Import the Groq client library
API_KEY ='YOUR_GROQ_API'   # used for authenticate requests
client = Groq(api_key=API_KEY)                                        # Groq client object for interacting with GROQ API
MODEL ="llama-3.1-8b-instant"                                         # LLM to use for generating responses
print(f'Groq client configured with model: {MODEL}')
print('Make sure API_KEY is replaced with your actual key!')

Groq client configured with model: llama-3.1-8b-instant
Make sure API_KEY is replaced with your actual key!




> FIRST API CALL



In [4]:
# Function to send to prompt to the LLM and get a response
def ask_llm(user_message, system_message="You are a helpful assistant", temperature=0.7, max_tokens=500):
  response = client.chat.completions.create(           # create a chat completion request
      model=MODEL,                                     # select the model

      messages=[                                       # define conversation messages
      {   "role":"system","content":system_message     # set the assistant's behavior
      },
      {   "role":"user","content":user_message         # user's prompt/question
      },
  ],
    temperature = temperature,
    max_tokens= max_tokens
 )
  return response.choices[0].message.content
test_response = ask_llm(
    "What is ETL in data engineering? Answer is exactly 2 sentence."
)
print('=== LLM  Response ===')
print(test_response)

=== LLM  Response ===
ETL (Extract, Transform, Load) is a data engineering process used to extract data from various sources, transform the data into a standardized format, and load it into a target system such as a data warehouse or database for analysis. ETL is commonly used to integrate data from different sources, ensure data consistency and quality, and provide a unified view of the data for business intelligence and analytics purposes.


In [5]:
# to ask questions easier

response_elt = ask_llm(
    "In 3 bullet points,explain how the Medallion Architecture"
    "(Bronze,Silver,Gold_layers) realtes to ELT pipeline.",
    system_message="You are a senior data engineering Instructor."
                   "Be concise and practical."
)
print('Medallion + ELT connection:')
print(response_elt)
print()
print('--- Token explanation ---')
print('Each word is roughly 1-2 tokens.')
print('The model above used approximately',len(response_elt.split())*1.3,'tokens.')
print('Llama-3.1-8b context window: 8192 tokens (-6000 words per conversation)')

Medallion + ELT connection:
Here are 3 key points explaining the Medallion Architecture (Bronze, Silver, Gold layers) and its relation to ELT (Extract, Load, Transform) pipeline:

* **Bronze Layer (Raw Data)**: This is the entry point of the ELT pipeline where raw data from various sources (e.g., databases, APIs) is **Extracted** through ETL (Extract, Transform, Load) processes. The Bronze layer captures data as-is, without any transformations, and stores it in a raw format.
* **Silver Layer (Processed Data)**: The Silver layer is where the **Transform** phase of ELT occurs. Data from the Bronze layer is processed, cleansed, and transformed into a more meaningful and structured format. This layer is often used for data aggregation, data quality checks, and data normalization.
* **Gold Layer (Curated Data)**: This is the final layer where **Load** phase of ELT occurs. The processed data from the Silver layer is loaded into a data warehouse or data mart, where it's optimized for analytic

# PROMPT ENGINEERING  
( improve AI responses)
## 1. Zero-Shot Prompting
Ask directly — no examples, no specific format instructions.

In [16]:
zero_shot_response = ask_llm(
    "Extract the city name from this address:"
    "456 Brigade Road, Bangalore 560025, Karnataka, India"
)
print('Zero-Shot Results')
print(zero_shot_response)
print()
ambiguous_response =  ask_llm('Clean this data: ramesh kumar, 45000, mumbai')
print('Ambiguous Zero-Shot Result:')
print(ambiguous_response)
print()
print('Problem: output format is unpredictable and not machine-parseable!')


Zero-Shot Results
The city name is: Bangalore.

Ambiguous Zero-Shot Result:
It seems like the data is a list of names and salaries, but it would be better if it was in a structured format. However, I can take a guess that the data is supposed to be:

- Name: ramesh kumar
- Salary: 45000
- Location: mumbai

If that's correct, I can reformat the data for you:

```
Name: ramesh kumar
Salary: 45000
Location: mumbai
```

If the data is supposed to be in a different format, please provide more context or information about how the data should be cleaned and structured.

Problem: output format is unpredictable and not machine-parseable!


## 2: Few-Shot Prompting
Provide 2 examples BEFORE the actual question. The model learns your format.

In [7]:
few_shot_prompt="""
Convert employee text to JSON. Here are examples:
Input: RAMESH KUMAR,45000,mumbai
output:{"name":"Ramesh Kumar","salary":45000,"city":"Mumbai"}

Input:priya nair, 52000,Delhi
output:{"name":"Priya Nair","salary":52000,"city":"Delhi"}

Now convert this:
Input:ANANYA DAS, 38000,kolkata
Output:"""

few_shot_response=ask_llm(
    few_shot_prompt, temperature=0.0)
print('Few-Shot Result:')
print(few_shot_response)
print()

try:
  parsed = json.loads(few_shot_response.strip())
  print('Successfully parsed as JSON!')
  print(f'Name:{parsed["name"]},Salary:{parsed['salary']},City:{parsed['city']}')
except json.JSONDecodeError:
        print('Parsing failed - model added extra text')
        print('Solution: add explicit instructions in the system prompt')



Few-Shot Result:
To convert the employee text to JSON, we can use the following Python code:

```python
import json

def convert_to_json(employee_text):
    # Split the input string into individual values
    values = employee_text.split(',')

    # Create a dictionary with the given keys
    employee = {
        "name": values[0].strip().title(),
        "salary": int(values[1].strip()),
        "city": values[2].strip().title()
    }

    # Convert the dictionary to JSON
    json_output = json.dumps(employee, indent=4)

    return json_output

# Test the function
employee_text = "ANANYA DAS, 30000, kolkata"
print(convert_to_json(employee_text))
```

When you run this code, it will output:

```json
{
    "name": "Ananya Das",
    "salary": 30000,
    "city": "Kolkata"
}
```

This code works by splitting the input string into individual values using the comma as a delimiter. It then creates a dictionary with the given keys and assigns the corresponding values. Finally, it converts the 

 ## 3: Role Prompting
 Tell the model who it is.

This changes the tone, depth, and vocabulary of the response

In [8]:
same_question ="Review this Python code and identify any issues:\n"
"df['revenue]=df['qty]*df['price]\n"
"result= df.groupby('dept').sum()"

generic_response = ask_llm(same_question, temperature=0.2)
print('Without Role Prompting:')
print(generic_response[:300],'...')
print()

role_response = ask_llm(
    same_question,
    system_message="You are a senior data engineer with 10 years of prediction"
    "experience. Review code critically for production readiness"
    "data type issues, and potential failures at scale.",
    temperature=0.2
)
print('with Role Prompting (Senior Data Engineer):')
print(role_response[:400],'...')
print()
print('Notice : role prompting produces more technical, actionable feedback')

Without Role Prompting:
I'm ready to help. Please go ahead and provide the Python code you'd like me to review. ...

with Role Prompting (Senior Data Engineer)
However, I don't see any code provided. Please paste the Python code you'd like me to review, and I'll do my best to identify any data type issues, potential failures at scale, and suggest improvements for production readiness.

Once you provide the code, I'll review it based on the following criteria:

1. **Data types**: Ensure that data types are correctly assigned and used throughout the code.
 ...

Notice : role prompting produces more technical, actionable feedback


## 4: Temperature Effect

In [9]:
prompt ="Give me one creative name for a data analytics startup"
print('=== Temperature Experiment ===')
for temp in [0.0,0.5,1.0]:
  response = ask_llm(prompt, temperature=temp)
  print(f'Temperature={temp}:{response.strip()}')
  time.sleep(1)
print()
print('Observation:')
print('  temperature=0.0 -> same or very similar answer every run(deterministic)')
print('  temperature=0.5 -> some variation')
print('  temperature=1.0 -> more creative/varied, sometimes surprising')
print()
print('Rule for data engineering tasks: use temperature =0.0 or 0.1')
print('You need CONSISTENT, PARSEABLE ouput - not creative variation')

=== Temperature Experiment ===
Temperature=0.0:Here's a creative name for a data analytics startup:

**Nexa Insights**

"Nexa" suggests connection and linkages, implying the ability to connect disparate data points and provide valuable insights. This name conveys the idea of a startup that helps businesses navigate complex data landscapes and uncover hidden patterns and trends.

Alternatively, if you'd like more options, I can provide you with a list of creative names for a data analytics startup.
Temperature=0.5:Here's a creative name for a data analytics startup:

**"Nexa Insights"**

"Nexa" is a combination of "nexus" (meaning connection or link) and "exa" (short for exabyte, a large unit of digital information). This name suggests that the startup helps connect people and businesses to valuable insights from vast amounts of data.
Temperature=1.0:Here's a creative name for a data analytics startup:

1. **Nexixa**: This name combines "nexus" (meaning a connection or link) and "ixa" (

## 5: Weak vs Strong Prompt Comparison
The most important experiment — same task, two different prompt qualities.

In [15]:
import json

invoice_text = (
    "Invoice #2024-001 from TECHWORLD SOLUTIONS dated 15th January 2024, Amount: Rs. 45,000 for Laptop"
)

weak_response = ask_llm(
    f"Clean this invoice data: {invoice_text}",
    temperature=0.3
)

print("WEAK PROMPT OUTPUT:")
print(weak_response)
print()

try:
    json.loads(weak_response)
    print("PARSEABLE: Yes")
except:
    print('PARSEABLE: No - Cannot load into DataFrame')

print('\n' + '='*50 + '\n')

strong_system ="""You are a data extraction specialist for an accounting pipeline.
Extract invoice data and return ONLY a valid JSON object,
Do NOT include any explanation, preamble, or markdown formatting.
Return ONLY the JSON, nothing else.

JSON schema (use null for missing values):
{"invoice_id": string,"vendor_name": string (Title Case),
 "amount":number(no currency symbols),
 "currency":string(default INR),
 "invoice_date": string (yyyy-mm-dd)
 "category": string (Electronics/Services/Accessories/Other)
 }
 """

strong_response = ask_llm(
    f"Extract from: {invoice_text}",
    system_message=strong_system,
    temperature=0.0
)

print("STRONG PROMPT OUTPUT:")
print(string_response)
print()
try:
  parsed= json.loads(strong_response.strip())
  print('PARSABLE: YES')
  print(f'Vendor: {parsed.get("vendor_name")}')
  print(f'Amount: {parsed.get("amount")}')
  print(f'Date: {parsed.get("invoice_date")}')
except json.JSONDecodeError :
  match = re.search(r'\{.*?\}', strong_response, re-DOTALL)
  if match:
    parsed = json.loads(match.group())
    print('PARSEABLE: Yes (extracted with regex fallback)')
  else:
    print('PARSEABLE: No- retry with stricter prompt')


WEAK PROMPT OUTPUT:
Here's the cleaned invoice data:

**Invoice Details:**

- **Invoice Number:** 2024-001
- **Date:** 15th January 2024
- **Vendor:** TECHWORLD SOLUTIONS
- **Description:** Laptop
- **Amount:** Rs. 45,000

Let me know if you need any further assistance.

PARSEABLE: No - Cannot load into DataFrame


STRONG PROMPT OUTPUT:
{"invoice_id": "2024-001","vendor_name": "Techworld Solutions","amount": 45000,"currency": "INR","invoice_date": "2024-01-15","category": "Electronics"}

PARSABLE: YES
Vendor: Techworld Solutions
Amount: 45000
Date: 2024-01-15


# Practice Questions - Answers

### Q1: What is the difference between ML (Day 5) and Generative AI (Day 6)?

* **Machine Learning (ML):** Learns patterns from data to make predictions or classifications.
* **Generative AI:** Creates new content such as text, images, code, or summaries using learned patterns.

---

### Q2: What does `temperature=0.0` do in an LLM API call and when would you use it?

* Makes the model's output more deterministic and consistent.
* Used for tasks requiring reliable structured output such as JSON extraction, data cleaning, and ETL pipelines.

---

### Q3: Write a few-shot prompt that extracts name and salary from text in JSON format.

```text
Input: Ramesh Kumar, 45000
Output: {"name":"Ramesh Kumar","salary":45000}

Input: Priya Nair, 52000
Output: {"name":"Priya Nair","salary":52000}

Input: Ananya Das, 38000
Output:
```

---

### Q4: What is LLM hallucination and how can prompt engineering reduce it?

* **Hallucination:** When an LLM generates incorrect or made-up information.
* **Reduction Methods:** Use clear instructions, provide examples (few-shot prompting), define output formats, and restrict responses to available data.

---

### Q5: Your LLM returns `json\n{"name":"Ramesh"}\n` and `json.loads()` crashes. Write the fix.

````python
response = response.replace("```json", "").replace("```", "").strip()
data = json.loads(response)
````

---

### Q6: How does today's Smart Data Cleaner differ from the manual ETL cleaning on Day 3?

| Manual ETL Cleaning  | Smart Data Cleaner (LLM)    |
| -------------------- | --------------------------- |
| Rule-based cleaning  | AI-based cleaning           |
| Fixed logic          | Understands context         |
| Requires custom code | Uses prompts                |
| Less flexible        | Handles varied text formats |


---
## MINI PROJECT: Smart Data Cleaner

**Goal:** Convert 5 messy invoice strings into a clean, structured Pandas DataFrame using LLM.

This is a complete GenAI-powered ETL pipeline:

In [18]:
from groq import Groq
import json
import pandas as pd

In [ ]:
API_KEY ="YOUR_GROQ_API"
client = Groq(api_key=API_KEY)
MODEL = "llama-3.1-8b-instant"

In [19]:
def ask_llm(user_message, system_message="You are a helful assistant", temperature=0.0,max_tokens=500):
  response=client.chat.completions.create(
      model=MODEL,
      messages=[
          {"role":"system","content":system_message},
          {"role":"user" ,"content":user_message}
      ],
      temperature=temperature,
      max_tokens=max_tokens
  )
  return response.choices[0].message.content

In [20]:
invoices=[
    "Invoice #2024-001 from TECHWORLD SOLUTIONS dated 15th January 2024, Amount: Rs. 45000 for Laptop",
    "Bill No INV-2024-002 | Vendor: office mart | Date 20/01/2024 | Total INR 12000 | Printer",
    "Payment Receipt #003 ABC Electronics 25-Jan-2024 Rs 8500 Mouse and Keyboard",
    "Invoice Number 2024-004 Vendor Smart Systems Amount ₹67000 Date 2024-02-01 Category Computer",
    "INV005 issued by Digital Hub on Feb 5 2024 total 15000 rupees for accessories"
]

In [22]:
system_prompt = """
You are a data extraction specialist.
Return ONLY valid JSON.
schema:{
"invoice_id":"",
"vendor_name":"",
"amount":0,
"invoice_date":"",
"category":""}
Do not return explanations.
Do not return markdown.
only return JSON"""

In [24]:
records =[]
for invoice in invoices:
  response = ask_llm(f"Extract data from:\n{invoice}",
                     system_message=system_prompt,
                     temperature=0.0
                     )
  print(response)
  try:
    parsed = json.loads(response)
    records.append(parsed)
  except:
    print("Failed to parse invoice")

{
  "invoice_id": "2024-001",
  "vendor_name": "TECHWORLD SOLUTIONS",
  "amount": 45000,
  "invoice_date": "2024-01-15",
  "category": "Laptop"
}
{
"invoice_id": "INV-2024-002",
"vendor_name": "office mart",
"amount": 12000,
"invoice_date": "20/01/2024",
"category": "Printer"
}
{
  "invoice_id": "003",
  "vendor_name": "ABC Electronics",
  "amount": 8500,
  "invoice_date": "25-Jan-2024",
  "category": "Mouse and Keyboard"
}
{
  "invoice_id": "2024-004",
  "vendor_name": "Smart Systems",
  "amount": 67000,
  "invoice_date": "2024-02-01",
  "category": "Computer"
}
{
  "invoice_id": "INV005",
  "vendor_name": "Digital Hub",
  "amount": 15000,
  "invoice_date": "Feb 5 2024",
  "category": "accessories"
}


In [25]:
df= pd.DataFrame(records)
print(df)

     invoice_id          vendor_name  amount invoice_date            category
0      2024-001  TECHWORLD SOLUTIONS   45000   2024-01-15              Laptop
1  INV-2024-002          office mart   12000   20/01/2024             Printer
2           003      ABC Electronics    8500  25-Jan-2024  Mouse and Keyboard
3      2024-004        Smart Systems   67000   2024-02-01            Computer
4        INV005          Digital Hub   15000   Feb 5 2024         accessories


In [26]:
# Total Revenue
print("Total Amount:")
print(df['amount'].sum())

Total Amount:
147500


In [27]:
# Average Invoice Value
print("Average Amount:")
print(df['amount'].mean())

Average Amount:
29500.0


In [30]:
print(df.sort_values(by='amount',ascending=False).head(1))

  invoice_id    vendor_name  amount invoice_date  category
3   2024-004  Smart Systems   67000   2024-02-01  Computer


In [32]:
category_analysis = (
    df.groupby("category")["amount"].sum().reset_index()
)
print(category_analysis)

             category  amount
0            Computer   67000
1              Laptop   45000
2  Mouse and Keyboard    8500
3             Printer   12000
4         accessories   15000


--END OF MINIPROJECT DAY-6--